# Indexing 2: point-by-point indexing (10 degree mosaicity)

Point-by-point (pbp) indexing with ImageD11: for every voxel of a grid in the
sample plane, the peaks whose diffraction path passes through that voxel are
indexed separately, and up to `gmax` candidate grains are kept per voxel.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys
from pathlib import Path

# make the repository importable, whatever the working directory is
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "simtools").is_dir())
sys.path.insert(0, str(REPO))

import os
import time

import numpy as np
import matplotlib.pyplot as plt
import ImageD11.cImageD11
import ImageD11.grain
import ImageD11.unitcell
import ImageD11.sinograms.point_by_point as pbp
from ImageD11.nbGui import nb_utils as utils
from simtools import dummy, io, paths, plot

plot.dark()

In [ ]:
SAMPLE = "domains_mosaicity_10p0deg"
colf = io.read_peaks(paths.peaks_path(SAMPLE))
print(f"{colf.nrows} peaks")

ypositions, ystep, ymin, y0 = dummy.add_pbp_columns(colf)
ny = len(ypositions)
nomega = len(np.unique(colf.omega))
print(f"{ny} translations with peaks, step {ystep:.5f} um, rotation axis at y0 = {y0}")

In [ ]:
pars = colf.parameters.parameters
ucell = ImageD11.unitcell.unitcell(
    [pars["cell__a"], pars["cell__b"], pars["cell__c"], pars["cell_alpha"], pars["cell_beta"], pars["cell_gamma"]],
    pars["cell_lattice_[P,A,B,C,I,F,R]"],
)
ucell.makerings(colf.ds.max())

In [ ]:
savefile = os.path.join(paths.process_dir(SAMPLE), "peaks_2d")
dset = dummy.dset(colf.copy(), paths.PARFILE, savefile, ny, nomega)
assert dset.ymin == ymin
assert dset.ymax == ypositions[-1]
assert np.allclose(dset.ystep, ystep)
assert np.allclose(dset.ostep, 0.5)
dset.save()

## Parameters

In [ ]:
percent_peaks_to_use = 99.99999  # strongest peaks used to generate trial orientations
eta_wedge = 6  # peaks within this many degrees of eta = 0 / 180 are not used
etacut = np.abs(np.sin(np.radians(eta_wedge)))
ifrac = np.percentile(colf.sum_intensity, 100 - percent_peaks_to_use) / colf.sum_intensity.max()
print(f"Using {percent_peaks_to_use}% of the peaks with intensity > {ifrac:.5f} * Imax")

pbpmanager = pbp.PBP(
    dset.parfile,
    dset,
    hkl_tol=0.07,  # peaks within this distance of integer hkl are assigned to a grain
    fpks=0.2,  # fraction of the predicted peaks a trial grain needs
    ds_tol=0.01,  # tolerance (1/A) on the ring assignment
    etacut=etacut,
    ifrac=ifrac,
    gmax=40,  # candidate grains kept per voxel
    cosine_tol=np.cos(np.radians(90 - dset.ostep)),
    y0=y0,
    symmetry="cubic",
    foridx=list(range(99)),  # rings used for indexing
    forgen=[0, 1],  # rings used to generate trial orientations
    uniqcut=0.2,
)
pbpmanager.setpeaks(colf.copy())

## Run

In [ ]:
ImageD11.cImageD11.cimaged11_omp_set_num_threads(1)
grains_filename = savefile + "_pbpmap.h5"
if os.path.isfile(grains_filename):
    os.remove(grains_filename)
t0 = time.perf_counter()
pbpmanager.point_by_point(grains_filename, loglevel=3, gridstep=1, nprocs=len(os.sched_getaffinity(0)))
print(f"done in {time.perf_counter() - t0:.0f} s")

## Result

In [ ]:
pbpmap = pbp.PBPMap(grains_filename)
pbpmap.plot_nuniq_hist()
pbpmap.choose_best(minpeaks=60)

fig, ax = plt.subplots(1, 1, figsize=(9, 9))
im = ax.pcolormesh(dset.sx_grid, dset.sy_grid, pbpmap.best_nuniq)
ax.set_aspect("equal")
ax.set_xlabel("x (um)")
ax.set_ylabel("y (um)")
ax.set_title("unique peaks of the best grain per voxel")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plot.despine(ax)
plt.show()

In [ ]:
m = ~np.isnan(pbpmap.best_ubi[:, :, 0, 0])
grains = []
for ubi in pbpmap.best_ubi[m]:
    g = ImageD11.grain.grain(ubi)
    g.ref_unitcell = ucell
    grains.append(g)
utils.get_rgbs_for_grains(grains)
rgbmap = np.zeros((ny, ny, 3))
rgbmap[m] = [g.rgb_z for g in grains]

fig, ax = plt.subplots(1, 1, figsize=(9, 9))
ax.pcolormesh(dset.sx_grid, dset.sy_grid, rgbmap)
ax.set_aspect("equal")
ax.set_xlabel("x (um)")
ax.set_ylabel("y (um)")
ax.set_title("IPF-z of the best grain per voxel")
plot.despine(ax)
plt.show()